# Option-Model Visual Report

This notebook assembles a compact, presentation-ready set of figures for
one option position: Black–Scholes call and put payoffs, a value profile,
extrinsic value, standardized Greeks, price and extrinsic-value surfaces, a
binomial lattice put, a SABR volatility smile, and a strategy payoff — all
built from the same underlying market inputs.

**Sections:**
1. Build the models
2. Numerical diagnostics
3. The figure set


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
from abaquant.derivatives import OptionStrategy
from abaquant.derivatives.models import BlackScholesMertonModel, CoxRossRubinsteinModel
from abaquant.derivatives.models.sabr import SABRVolatilityModel
from abaquant.visualization import VisualizationError

## 1. Build the models

A Black–Scholes model, a binomial lattice, and a SABR smile — all sharing spot=100, strike=105, T=1y.


In [ ]:
models = {
    "black_scholes": BlackScholesMertonModel(100.0, 105.0, 1.0, 0.05, 0.22),
    "lattice": CoxRossRubinsteinModel(100.0, 105.0, 1.0, 0.05, 0.22, number_of_steps=8),
    "sabr": SABRVolatilityModel(100.0, 105.0, 1.0, 0.20, 0.7, -0.25, 0.45),
}

## 2. Numerical diagnostics

The scalar values that correspond to the figures below.


In [ ]:
report_values = {
    "bsm_call_price": models["black_scholes"].call_price(),
    "bsm_put_price": models["black_scholes"].put_price(),
    "bsm_call_extrinsic_value": models["black_scholes"].extrinsic_value("call"),
    "bsm_call_moneyness": models["black_scholes"].moneyness(),
    "bsm_call_forward_moneyness": models["black_scholes"].forward_moneyness(),
    "bsm_call_break_even_price": models["black_scholes"].break_even_price("call"),
    "lattice_put_price": models["lattice"].put_price(),
    "sabr_atm_iv": models["sabr"].implied_vol(),
}
for key, value in report_values.items():
    print(f"{key:30s}: {value:.6f}")

## 3. The figure set

Eleven figures covering payoff, value profile, extrinsic decomposition, Greeks, surfaces, the lattice, the SABR smile, and a strategy payoff.


In [ ]:
try:
    figures = {
        "call_payoff": models["black_scholes"].visualize(chart="payoff", option_type="call"),
        "put_payoff": models["black_scholes"].visualize(chart="payoff", option_type="put"),
        "call_value_profile": models["black_scholes"].visualize(
            chart="price_profile", option_type="call"
        ),
        "call_extrinsic_profile": models["black_scholes"].visualize(
            chart="extrinsic_value", option_type="call"
        ),
        "call_greeks": models["black_scholes"].visualize(
            chart="greeks", option_type="call", greek_scale="standardized"
        ),
        "call_price_surface": models["black_scholes"].visualize(
            chart="price_surface", option_type="call", grid_size=31, volatility_grid_size=15
        ),
        "call_extrinsic_surface": models["black_scholes"].visualize(
            chart="extrinsic_surface", option_type="call", grid_size=31, volatility_grid_size=15
        ),
        "put_lattice": models["lattice"].visualize(chart="tree", option_type="put"),
        "sabr_smile": models["sabr"].visualize(chart="volatility_smile"),
        "strategy_payoff": OptionStrategy.bull_call_spread(
            lower_strike=100.0, upper_strike=115.0, lower_premium=6.0, upper_premium=2.0
        ).visualize(chart="payoff"),
    }
    print(f"Created {len(figures)} figures: {list(figures)}")
except VisualizationError as exc:
    print(f"Visualization skipped (optional dependency missing): {exc}")

## Takeaway

This is a good template for a one-position "tear sheet." Combine it with
`model.report(option_type=...)` (see notebook **21 — Exportable Reports**)
to turn the same diagnostics into a Markdown/HTML/PDF deliverable.
